In [5]:
!pip install pyspark -q

import os
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, when, sum as _sum, round

spark = SparkSession.builder \
    .appName("SmartHomeEnergyColab") \
    .master("local[*]") \
    .getOrCreate()

print("Spark session active!")

uploaded = files.upload()



Spark session active!


Saving large_sensor_logs.csv to large_sensor_logs.csv


In [8]:
# 1. Extract the filename string from the uploaded dictionary
uploaded_filename = list(uploaded.keys())[0]

# 2. Read the CSV using the string path
df = spark.read.csv(uploaded_filename, header=True, inferSchema=True)

print("--- 1. Raw Telemetry Schema ---")
df.printSchema()

print("\n--- 2. Sample Data ---")
df.show(5)

# 3. Classify Peak (06:00 to 22:00) vs Off-Peak
df_classified = df.withColumn("reading_hour", hour(col("timestamp"))) \
                  .withColumn(
                      "time_of_day",
                      when((col("reading_hour") >= 6) & (col("reading_hour") < 22), "PEAK")
                      .otherwise("OFF_PEAK")
                  )

# 4. Top Energy Consuming Devices
top_devices = df_classified.groupBy("device_id", "device_type") \
    .agg(round(_sum("energy_kwh"), 2).alias("total_kwh")) \
    .orderBy(col("total_kwh").desc())

print("\n--- 3. Top Energy Consuming Devices ---")
top_devices.show()

# 5. Usage Breakdown: Peak vs Off-Peak
peak_summary = df_classified.groupBy("device_id", "time_of_day") \
    .agg(round(_sum("energy_kwh"), 2).alias("kwh")) \
    .orderBy("device_id")

print("\n--- 4. Usage Breakdown: Peak vs Off-Peak ---")
peak_summary.show()

# 6. Export summary to a clean CSV in Colab session storage
output_filename = "output_top_devices.csv"
top_devices.toPandas().to_csv(output_filename, index=False)
print(f"\n[SUCCESS] Aggregation complete and saved to '{output_filename}'")

--- 1. Raw Telemetry Schema ---
root
 |-- device_id: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- room_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage: double (nullable = true)


--- 2. Sample Data ---
+-----------+-----------+-------+-------------------+----------+-------+
|  device_id|device_type|room_id|          timestamp|energy_kwh|voltage|
+-----------+-----------+-------+-------------------+----------+-------+
|DEV_HVAC_01|       HVAC|      1|2026-10-08 02:00:00|      0.45|  230.1|
|DEV_HVAC_01|       HVAC|      1|2026-10-08 07:30:00|      1.85|  229.4|
|DEV_HVAC_01|       HVAC|      1|2026-10-08 14:15:00|       2.4|  228.9|
|DEV_HVAC_01|       HVAC|      1|2026-10-08 20:00:00|      2.15|  231.0|
|DEV_HVAC_01|       HVAC|      1|2026-10-08 23:30:00|      0.52|  230.5|
+-----------+-----------+-------+-------------------+----------+-------+
only showing top 5 rows

--- 3. 